# Phase 2 — Step 4: Asymmetric Hybrid Fusion

## Objective
Achieve **absolute security** in the hybrid retriever by enforcing RBAC metadata filtering at the engine level, eliminating all security breaches from Steps 1–3.

## The Problem (from previous steps)
Steps 1–3 demonstrated that BM25 has no native metadata filtering — it returns documents purely by lexical relevance, ignoring `clearance_level` and `allowed_departments`. This causes security breaches where unauthorized users receive sensitive PII chunks.

## Architecture: Asymmetric Pipeline

```
Query → Intent Router (α) ─┬─→ ChromaDB Branch: native WHERE pre-filter → vector search → Top K secure
                            │
                            └─→ BM25 Branch: lexical search → Top N pool → Python RBAC post-filter → Top K secure
                                                                    │
                                                         ┌──────────┘
                                                         ▼
                              Secure RRF Fusion (only clean docs) → Final Top 5
```

**Key design asymmetry:**
- **ChromaDB**: Applies `where` clause BEFORE vector search (mathematically eliminates unauthorized docs from the search space).
- **BM25**: Fetches an oversized pool (Top 30), then a Python loop removes unauthorized chunks. We backfill to ensure K surviving results.

## Metrics
- **Security Breach Rate**: Must be 0% for all test cases.
- **Latency**: Measure ms penalty of the post-filtering loop vs. unsecured baseline.
- **MRR / Relevant@K**: Verify retrieval quality is maintained despite filtering.

In [1]:
# Cell 1 — Imports & Configuration
import json, time, re, os
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb

BASE = os.path.dirname(os.path.abspath("__file__"))
RESULTS_DIR = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2")
CHUNK_FILE = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph1", "chunk_results.json")

K = 5                  # final top-K
BM25_POOL = 30         # oversized BM25 pool for post-filtering
N_WARMUP = 3           # warmup runs for latency measurement
N_BENCH = 10           # benchmark iterations per query

print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")
print(f"Chunk file:  {os.path.abspath(CHUNK_FILE)}")
print("Step 4: Asymmetric Hybrid Fusion — ready.")

Results dir: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\data\results\notebook_results\ph2
Chunk file:  D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\data\results\notebook_results\ph1\chunk_results.json
Step 4: Asymmetric Hybrid Fusion — ready.


In [2]:
# Cell 2 — Load corpus + indexing

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    all_strategies = json.load(f)

corpus: list[Document] = []
for src_file, chunks in all_strategies["custom_rbac"].items():
    for ch in chunks:
        meta = ch["metadata"].copy()
        meta.setdefault("contains_PII", False)
        meta.setdefault("sensitivity_types", [])
        corpus.append(Document(page_content=ch["page_content"], metadata=meta))
print(f"Custom RBAC corpus: {len(corpus)} chunks")

# --- Embedding model ---
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

# --- ChromaDB ---
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="step4_asym", metadata={"hnsw:space": "cosine"})

def sanitize(meta: dict) -> dict:
    return {k: (json.dumps(v) if isinstance(v, list) else v if isinstance(v, (str, int, float, bool)) else str(v))
            for k, v in meta.items()}

print("Computing embeddings...")
embs = embedding_model.embed_documents([d.page_content for d in corpus])
collection.add(
    documents=[d.page_content for d in corpus],
    embeddings=embs,
    ids=[f"c{i:03d}" for i in range(len(corpus))],
    metadatas=[sanitize(d.metadata) for d in corpus]
)
print(f"ChromaDB: {collection.count()} docs")

# --- BM25 ---
def tokenize(t: str) -> list[str]:
    return re.findall(r"\w+", t.lower())

tok_corpus = [tokenize(d.page_content) for d in corpus]
bm25 = BM25Okapi(tok_corpus)
print(f"BM25: {len(tok_corpus)} docs")

Custom RBAC corpus: 33 chunks
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Computing embeddings...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


ChromaDB: 33 docs
BM25: 33 docs


## Cell 3 — Core: Unsecured Baseline vs. Secure Asymmetric Retriever

Two pipelines implemented side-by-side:

1. **Unsecured Baseline** (`ens_rrf_unsecured`): Standard RRF — no filtering at any stage. Identical to Step 1.
2. **Secure Asymmetric** (`ens_rrf_secure`):
   - ChromaDB: `where` clause filters `clearance_level <= user_cl`. For restricted chunks (cl≥2), also requires department match.
   - BM25: Fetches Top-30 pool → Python RBAC loop removes unauthorized → surviving docs enter RRF.
   - RRF applied only on clean, authorized documents.

In [3]:
# Cell 3 — Retrieval engines: Unsecured Baseline + Secure Asymmetric

@dataclass
class RR:
    """Retrieval Result."""
    document: Document
    score: float
    source_engine: str
    rank: int = 0

# ──────────────────────────────────────────────────────────
# UNSECURED engines (baseline, identical to Step 1)
# ──────────────────────────────────────────────────────────

def chroma_unsecured(q: str, k: int = 5) -> list[RR]:
    """Semantic retrieval — NO metadata filtering."""
    qe = embedding_model.embed_query(q)
    r = collection.query(query_embeddings=[qe], n_results=k,
                         include=["documents", "metadatas", "distances"])
    return [RR(Document(page_content=r["documents"][0][i], metadata=r["metadatas"][0][i]),
               1.0 - r["distances"][0][i], "chroma", i + 1)
            for i in range(len(r["documents"][0]))]

def bm25_unsecured(q: str, k: int = 5) -> list[RR]:
    """Lexical retrieval — NO metadata filtering."""
    sc = bm25.get_scores(tokenize(q))
    top = np.argsort(sc)[::-1][:k]
    return [RR(corpus[i], float(sc[i]), "bm25", r + 1)
            for r, i in enumerate(top) if sc[i] > 0]

def ens_rrf_unsecured(q: str, k: int = 5, alpha: float = 0.5, rk: int = 60) -> list[RR]:
    """Baseline RRF — no security filtering at any stage."""
    cr = chroma_unsecured(q, k * 2)
    br = bm25_unsecured(q, k * 2)
    sc, dm, sm = {}, {}, {}
    for r in cr:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + alpha * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("chroma")
    for r in br:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + (1 - alpha) * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("bm25")
    sk = sorted(sc, key=sc.get, reverse=True)[:k]
    return [RR(dm[key], sc[key], "ens(" + "+".join(sorted(set(sm[key]))) + ")", i + 1)
            for i, key in enumerate(sk)]

# ──────────────────────────────────────────────────────────
# SECURE ASYMMETRIC engines
# ──────────────────────────────────────────────────────────

def _check_rbac(user_cl: int, user_dept: str, meta: dict) -> bool:
    """RBAC check: clearance_level + department enforcement."""
    chunk_cl = int(meta.get("clearance_level", 0))
    chunk_dept = meta.get("allowed_departments", "all")
    if chunk_cl > user_cl:
        return False
    if chunk_cl >= 2 and chunk_dept != "all" and user_dept != chunk_dept:
        return False
    return True

def chroma_secure(q: str, user_cl: int, user_dept: str, k: int = 5) -> list[RR]:
    """Semantic retrieval with native WHERE pre-filtering.
    
    ChromaDB's where clause eliminates unauthorized documents from the
    vector search space BEFORE computing cosine similarity. This is 
    mathematically secure — unauthorized docs never enter the ranking.
    
    Filtering strategy:
    - clearance_level <= user_cl (always applied)
    - For dept-restricted chunks (cl>=2), we cannot express OR logic
      in a single ChromaDB where clause, so we query with clearance
      filter and post-verify department (minimal overhead since ChromaDB
      already reduced the candidate set).
    """
    qe = embedding_model.embed_query(q)
    
    # Native pre-filter: only chunks with clearance_level <= user's level
    where_filter = {"clearance_level": {"$lte": user_cl}}
    
    r = collection.query(
        query_embeddings=[qe],
        n_results=k * 2,  # fetch extra to compensate for dept filtering
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )
    
    results: list[RR] = []
    rank = 1
    for i in range(len(r["documents"][0])):
        meta = r["metadatas"][0][i]
        # Department check (for cl>=2 restricted chunks)
        if not _check_rbac(user_cl, user_dept, meta):
            continue
        results.append(RR(
            Document(page_content=r["documents"][0][i], metadata=meta),
            1.0 - r["distances"][0][i], "chroma_secure", rank
        ))
        rank += 1
        if len(results) >= k:
            break
    
    return results

def bm25_secure(q: str, user_cl: int, user_dept: str, k: int = 5,
                pool_size: int = BM25_POOL) -> list[RR]:
    """Lexical retrieval with Python RBAC post-filtering.
    
    BM25 has no native metadata filtering. Strategy:
    1. Fetch an oversized pool (Top-30) from BM25.
    2. Iterate through results, apply RBAC check on each.
    3. Keep only authorized chunks, stop at k.
    
    The pool_size must be large enough that after filtering,
    at least k results survive.
    """
    sc = bm25.get_scores(tokenize(q))
    top_indices = np.argsort(sc)[::-1][:pool_size]
    
    results: list[RR] = []
    filtered_count = 0
    rank = 1
    
    for idx in top_indices:
        if sc[idx] <= 0:
            break
        doc = corpus[idx]
        meta = doc.metadata
        
        if _check_rbac(user_cl, user_dept, meta):
            results.append(RR(doc, float(sc[idx]), "bm25_secure", rank))
            rank += 1
            if len(results) >= k:
                break
        else:
            filtered_count += 1
    
    return results

def ens_rrf_secure(q: str, user_cl: int, user_dept: str,
                   k: int = 5, alpha: float = 0.5, rk: int = 60) -> list[RR]:
    """Secure Asymmetric RRF: fuses ONLY authorized documents.
    
    Both branches deliver pre-cleaned results → RRF math operates
    exclusively on documents the user is authorized to see.
    """
    cr = chroma_secure(q, user_cl, user_dept, k * 2)
    br = bm25_secure(q, user_cl, user_dept, k * 2)
    
    sc, dm, sm = {}, {}, {}
    for r in cr:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + alpha * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("chroma")
    for r in br:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + (1 - alpha) * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("bm25")
    
    sk = sorted(sc, key=sc.get, reverse=True)[:k]
    return [RR(dm[key], sc[key], "secure_ens(" + "+".join(sorted(set(sm[key]))) + ")", i + 1)
            for i, key in enumerate(sk)]

print("Unsecured baseline + Secure asymmetric retrievers loaded.")

Unsecured baseline + Secure asymmetric retrievers loaded.


In [4]:
# Cell 4 — Intent Router (from Step 2 v2) + PII Detection + Audit

@dataclass
class IC:
    intent: str; alpha: float; patterns: list[str]; reasoning: str

T1 = [{"n": "password", "p": r"(?i)\b(?:password|contraseña|pwd|credential|override)\b"},
      {"n": "iban", "p": r"(?i)\bIBAN\b"}, {"n": "swift", "p": r"(?i)\bSWIFT\b"}]
T2 = [{"n": "client_id", "p": r"(?i)\bCLI-\d{3}\b"},
      {"n": "ip", "p": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b"},
      {"n": "email", "p": r"(?i)\b(?:email|e-mail|correo)\s*(?:address|de)?"},
      {"n": "log", "p": r"(?i)\b(?:log|logs|error\s+log|server\s+log)\b"},
      {"n": "account", "p": r"(?i)\b(?:account\s+number|bank\s+account)\b"},
      {"n": "name", "p": r"\b[A-Z][a-záéíóúñ]+\s+(?:Gómez|Ruiz|Mendoza|Torres|Silva)\b"},
      {"n": "id_seek", "p": r"(?i)\b(?:identifier|code|codi|código|número)\b"}]
T3 = [{"n": "how", "p": r"(?i)^(?:how|com|cómo)\s"},
      {"n": "what", "p": r"(?i)^(?:what|què|qué)\s(?:is|are|és)\b"},
      {"n": "describe", "p": r"(?i)\b(?:summary|overview|resum|explain|describe)\b"},
      {"n": "perf", "p": r"(?i)\b(?:performance|revenue|budget|rendiment|ingressos)\b"},
      {"n": "product", "p": r"(?i)\b(?:product|feature|kit|timer|contents|specifications)\b"},
      {"n": "quarter", "p": r"(?i)\bQ[1-4]\b"},
      {"n": "topic", "p": r"(?i)\b(?:about|sobre|regarding|strategy|forecast)\b"}]

def classify(q: str) -> IC:
    t1 = [p["n"] for p in T1 if re.search(p["p"], q)]
    t2 = [p["n"] for p in T2 if re.search(p["p"], q)]
    t3 = [p["n"] for p in T3 if re.search(p["p"], q)]
    allp, exact = t1 + t2 + t3, t1 + t2
    if t1: return IC("exact_critical", 0.2, allp, f"TIER1: {t1}")
    if t2 and not t3: return IC("exact", 0.2, allp, f"Exact: {t2}")
    if t3 and not exact: return IC("conceptual", 0.8, allp, f"Conceptual: {t3}")
    if exact and t3:
        if len(exact) * 1.5 >= len(t3): return IC("mixed_exact_lean", 0.35, allp, "Mixed exact-lean")
        return IC("mixed_concept_lean", 0.65, allp, "Mixed concept-lean")
    return IC("default", 0.5, [], "No signal")

# PII detection
PII_RE = {
    "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "iban": r"[A-Z]{2}\d{2}[\s]?[A-Z0-9]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}",
    "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    "password": r"(?i)(?:password|override[_ ]?password)\s*[:=]\s*['\"]?([^\s'\"]+)",
    "client_id": r"CLI-\d{3,}",
    "swift_code": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}",
    "person_name": r"\b[A-Z][a-záéíóúñ]+\s+(?:Gómez|Ruiz|Mendoza|Torres|Silva)\b",
    "monetary": r"€[\d.,]+\s*(?:million|netos)?",
}

def det_pii(t: str) -> dict:
    return {n: m for n, p in PII_RE.items() if (m := re.findall(p, t))}

def audit(results: list[RR], user_cl: int, user_dept: str) -> dict:
    """Full security audit of retrieval results."""
    breaches = 0
    leaked_pii: set[str] = set()
    for r in results:
        meta = r.document.metadata
        if not _check_rbac(user_cl, user_dept, meta):
            breaches += 1
            leaked_pii.update(det_pii(r.document.page_content).keys())
    return {
        "breaches": breaches,
        "leaked_pii": sorted(leaked_pii),
        "security": "PASS" if breaches == 0 else "FAIL",
        "total_pii_in_results": sorted(set().union(*(det_pii(r.document.page_content).keys() for r in results)) if results else set()),
    }

print("Intent router + PII detection + audit loaded.")

Intent router + PII detection + audit loaded.


In [5]:
# Cell 5 — Test Queries: Security validation + latency benchmarking

@dataclass
class TestCase:
    tid: str
    query: str
    user_cl: int
    user_dept: str
    description: str
    target_pii: list[str] = field(default_factory=list)
    relevant_files: list[str] = field(default_factory=list)
    relevant_kw: list[str] = field(default_factory=list)

TESTS = [
    # --- Critical security tests (unauthorized users seeking PII) ---
    TestCase("S1", "What is the admin override password for the backend?",
             0, "all", "Public user → password (cl=3 engineering)",
             ["password"], ["server_logs_witty_backend.txt"], ["password", "override"]),
    TestCase("S2", "IBAN bank account number for payments",
             0, "all", "Public user → IBAN (cl=3 legal)",
             ["iban", "swift_code"], ["distribution-contract-2026.docx"], ["IBAN", "SWIFT"]),
    TestCase("S3", "email addresses of clients Laura Gomez and Marcos Ruiz",
             0, "all", "Public user → client emails (cl=3 sales)",
             ["email", "client_id"], ["clients-and-billings.xlsx"], ["email", "CLI-"]),
    TestCase("S4", "What IP address caused the failed login in server logs?",
             0, "all", "Public user → IP address (cl=2 engineering)",
             ["ip_address"], ["server_logs_witty_backend.txt"], ["IP", "login", "failed"]),
    TestCase("S5", "IBAN bank account number for payments",
             1, "sales", "Sales intern cl=1 → IBAN (cl=3 legal)",
             ["iban", "swift_code"], ["distribution-contract-2026.docx"], ["IBAN", "SWIFT"]),

    # --- Authorized access tests (verify data is still accessible) ---
    TestCase("A1", "What is the admin override password for the backend?",
             3, "engineering", "Senior engineer → password (authorized)",
             ["password"], ["server_logs_witty_backend.txt"], ["password", "override"]),
    TestCase("A2", "IBAN bank account number for payments",
             3, "legal", "Legal counsel → IBAN (authorized)",
             ["iban", "swift_code"], ["distribution-contract-2026.docx"], ["IBAN", "SWIFT"]),
    TestCase("A3", "email addresses of clients Laura Gomez and Marcos Ruiz",
             3, "sales", "Sales manager → client emails (authorized)",
             ["email", "client_id"], ["clients-and-billings.xlsx"], ["email", "CLI-"]),

    # --- Cross-department tests ---
    TestCase("X1", "IBAN bank account number for payments",
             3, "engineering", "Engineer cl=3 → IBAN (legal-only)",
             ["iban"], ["distribution-contract-2026.docx"], ["IBAN"]),
    TestCase("X2", "email addresses of clients Laura Gomez and Marcos Ruiz",
             3, "finance", "Finance cl=3 → client emails (sales-only)",
             ["email"], ["clients-and-billings.xlsx"], ["email"]),

    # --- Public/conceptual queries (no filtering needed) ---
    TestCase("P1", "What are the contents of the Witty Kit?",
             0, "all", "Public user → product info (cl=0)",
             [], ["Witty-QuickGuide-EN.pdf"], ["kit", "content", "timer"]),
    TestCase("P2", "How do I switch on the Witty timer?",
             0, "all", "Public user → usage instructions (cl=0)",
             [], ["Witty-QuickGuide-EN.pdf"], ["switch", "on", "button", "timer"]),
]

print(f"{len(TESTS)} test cases defined.")
for t in TESTS:
    print(f"  {t.tid}: cl={t.user_cl} dept={t.user_dept:12s} — {t.description}")

12 test cases defined.
  S1: cl=0 dept=all          — Public user → password (cl=3 engineering)
  S2: cl=0 dept=all          — Public user → IBAN (cl=3 legal)
  S3: cl=0 dept=all          — Public user → client emails (cl=3 sales)
  S4: cl=0 dept=all          — Public user → IP address (cl=2 engineering)
  S5: cl=1 dept=sales        — Sales intern cl=1 → IBAN (cl=3 legal)
  A1: cl=3 dept=engineering  — Senior engineer → password (authorized)
  A2: cl=3 dept=legal        — Legal counsel → IBAN (authorized)
  A3: cl=3 dept=sales        — Sales manager → client emails (authorized)
  X1: cl=3 dept=engineering  — Engineer cl=3 → IBAN (legal-only)
  X2: cl=3 dept=finance      — Finance cl=3 → client emails (sales-only)
  P1: cl=0 dept=all          — Public user → product info (cl=0)
  P2: cl=0 dept=all          — Public user → usage instructions (cl=0)


In [6]:
# Cell 6 — Execute: Unsecured vs Secure side-by-side + latency benchmark

def compute_mrr(results: list[RR], relevant_files: list[str], relevant_kw: list[str]) -> float:
    """MRR based on source_file match + keyword presence."""
    for r in results:
        meta = r.document.metadata
        src = meta.get("source_file", "")
        text_lower = r.document.page_content.lower()
        file_match = any(f in src for f in relevant_files)
        kw_match = any(kw.lower() in text_lower for kw in relevant_kw) if relevant_kw else True
        if file_match and kw_match:
            return 1.0 / r.rank
    return 0.0

all_results = []

for tc in TESTS:
    print("=" * 95)
    print(f"  {tc.tid}: {tc.description}")
    print(f"  Query: \"{tc.query}\"")
    print(f"  User: cl={tc.user_cl}, dept={tc.user_dept}")
    print("=" * 95)

    ic = classify(tc.query)
    print(f"  Intent: {ic.intent} (α={ic.alpha})")

    # ── UNSECURED BASELINE ──
    # Warmup
    for _ in range(N_WARMUP):
        ens_rrf_unsecured(tc.query, K, ic.alpha)
    # Benchmark
    t_unsec = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        unsec_results = ens_rrf_unsecured(tc.query, K, ic.alpha)
        t_unsec.append((time.perf_counter() - t0) * 1000)
    lat_unsec = np.median(t_unsec)
    unsec_audit = audit(unsec_results, tc.user_cl, tc.user_dept)
    unsec_mrr = compute_mrr(unsec_results, tc.relevant_files, tc.relevant_kw)

    print(f"\n  ── UNSECURED BASELINE ({len(unsec_results)} chunks, {lat_unsec:.1f}ms median) ──")
    for i, r in enumerate(unsec_results):
        m = r.document.metadata
        pii = det_pii(r.document.page_content)
        pii_s = ", ".join(pii.keys()) if pii else "-"
        ok = "OK" if _check_rbac(tc.user_cl, tc.user_dept, m) else "BREACH"
        txt = r.document.page_content[:60].replace("\n", " ")
        print(f"    #{i+1} {m.get('chunk_id','?'):12s} cl={m.get('clearance_level',0)} "
              f"dept={str(m.get('allowed_departments','all')):12s} [{ok:6s}] PII={pii_s}")
    print(f"  Audit: {unsec_audit['security']} | breaches={unsec_audit['breaches']} | "
          f"leaked={unsec_audit['leaked_pii']} | MRR={unsec_mrr:.2f}")

    # ── SECURE ASYMMETRIC ──
    # Warmup
    for _ in range(N_WARMUP):
        ens_rrf_secure(tc.query, tc.user_cl, tc.user_dept, K, ic.alpha)
    # Benchmark
    t_sec = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        sec_results = ens_rrf_secure(tc.query, tc.user_cl, tc.user_dept, K, ic.alpha)
        t_sec.append((time.perf_counter() - t0) * 1000)
    lat_sec = np.median(t_sec)
    sec_audit = audit(sec_results, tc.user_cl, tc.user_dept)
    sec_mrr = compute_mrr(sec_results, tc.relevant_files, tc.relevant_kw)

    print(f"\n  ── SECURE ASYMMETRIC ({len(sec_results)} chunks, {lat_sec:.1f}ms median) ──")
    for i, r in enumerate(sec_results):
        m = r.document.metadata
        pii = det_pii(r.document.page_content)
        pii_s = ", ".join(pii.keys()) if pii else "-"
        txt = r.document.page_content[:60].replace("\n", " ")
        print(f"    #{i+1} {m.get('chunk_id','?'):12s} cl={m.get('clearance_level',0)} "
              f"dept={str(m.get('allowed_departments','all')):12s} [OK    ] PII={pii_s}")
    print(f"  Audit: {sec_audit['security']} | breaches={sec_audit['breaches']} | "
          f"PII in results={sec_audit['total_pii_in_results']} | MRR={sec_mrr:.2f}")

    # ── DELTA ──
    lat_delta = lat_sec - lat_unsec
    lat_pct = (lat_delta / lat_unsec * 100) if lat_unsec > 0 else 0
    print(f"\n  Δ Latency: {lat_delta:+.1f}ms ({lat_pct:+.0f}%) | "
          f"Δ Breaches: {unsec_audit['breaches']} → {sec_audit['breaches']} | "
          f"Δ MRR: {unsec_mrr:.2f} → {sec_mrr:.2f}")

    all_results.append({
        "tid": tc.tid, "query": tc.query,
        "user_cl": tc.user_cl, "user_dept": tc.user_dept,
        "description": tc.description, "intent": ic.intent, "alpha": ic.alpha,
        "unsec_chunks": len(unsec_results), "sec_chunks": len(sec_results),
        "unsec_breaches": unsec_audit["breaches"], "sec_breaches": sec_audit["breaches"],
        "unsec_leaked_pii": ", ".join(unsec_audit["leaked_pii"]),
        "sec_leaked_pii": "",  # should always be empty
        "unsec_security": unsec_audit["security"], "sec_security": sec_audit["security"],
        "unsec_mrr": round(unsec_mrr, 4), "sec_mrr": round(sec_mrr, 4),
        "unsec_lat_ms": round(lat_unsec, 2), "sec_lat_ms": round(lat_sec, 2),
        "lat_delta_ms": round(lat_delta, 2), "lat_delta_pct": round(lat_pct, 1),
    })
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  S1: Public user → password (cl=3 engineering)
  Query: "What is the admin override password for the backend?"
  User: cl=0, dept=all
  Intent: exact_critical (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 9.6ms median) ──
    #1 custom_005   cl=3 dept=engineering  [BREACH] PII=password
    #2 custom_003   cl=2 dept=engineering  [BREACH] PII=-
    #3 custom_001   cl=0 dept=all          [OK    ] PII=-
    #4 custom_004   cl=2 dept=engineering  [BREACH] PII=-
    #5 custom_002   cl=0 dept=all          [OK    ] PII=-
  Audit: FAIL | breaches=3 | leaked=['password'] | MRR=1.00



  ── SECURE ASYMMETRIC (2 chunks, 8.9ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=[] | MRR=0.00

  Δ Latency: -0.7ms (-7%) | Δ Breaches: 3 → 0 | Δ MRR: 1.00 → 0.00

  S2: Public user → IBAN (cl=3 legal)
  Query: "IBAN bank account number for payments"
  User: cl=0, dept=all
  Intent: exact_critical (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 8.3ms median) ──
    #1 custom_004   cl=3 dept=legal        [BREACH] PII=iban, swift_code
    #2 custom_003   cl=2 dept=engineering  [BREACH] PII=-
    #3 custom_002   cl=0 dept=all          [OK    ] PII=-
    #4 custom_002   cl=3 dept=finance      [BREACH] PII=monetary
    #5 custom_004   cl=3 dept=finance      [BREACH] PII=monetary
  Audit: FAIL | breaches=4 | leaked=['iban', 'monetary', 'swift_code'] | MRR=1.00



  ── SECURE ASYMMETRIC (2 chunks, 8.7ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=[] | MRR=0.00

  Δ Latency: +0.4ms (+4%) | Δ Breaches: 4 → 0 | Δ MRR: 1.00 → 0.00

  S3: Public user → client emails (cl=3 sales)
  Query: "email addresses of clients Laura Gomez and Marcos Ruiz"
  User: cl=0, dept=all
  Intent: exact (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 9.0ms median) ──
    #1 custom_003   cl=3 dept=sales        [BREACH] PII=email, client_id, person_name
    #2 custom_001   cl=3 dept=sales        [BREACH] PII=email, client_id, person_name
    #3 custom_000   cl=3 dept=sales        [BREACH] PII=-
    #4 custom_002   cl=3 dept=finance      [BREACH] PII=monetary
    #5 custom_003   cl=3 dept=finance      [BREACH] PII=-
  Audit: FAIL | breaches=5 | leaked=['client_id', 'email', 'monetary', 'person_name'] | MRR=1.00



  ── SECURE ASYMMETRIC (2 chunks, 8.9ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=[] | MRR=0.00

  Δ Latency: -0.2ms (-2%) | Δ Breaches: 5 → 0 | Δ MRR: 1.00 → 0.00

  S4: Public user → IP address (cl=2 engineering)
  Query: "What IP address caused the failed login in server logs?"
  User: cl=0, dept=all
  Intent: exact (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 9.0ms median) ──
    #1 custom_003   cl=2 dept=engineering  [BREACH] PII=-
    #2 custom_004   cl=2 dept=engineering  [BREACH] PII=-
    #3 custom_002   cl=2 dept=engineering  [BREACH] PII=ip_address
    #4 custom_003   cl=3 dept=finance      [BREACH] PII=-
    #5 custom_001   cl=3 dept=finance      [BREACH] PII=-
  Audit: FAIL | breaches=5 | leaked=['ip_address'] | MRR=1.00


...


  ── SECURE ASYMMETRIC (5 chunks, 13.7ms median) ──
    #1 custom_003   cl=3 dept=sales        [OK    ] PII=email, client_id, person_name
    #2 custom_001   cl=3 dept=sales        [OK    ] PII=email, client_id, person_name
    #3 custom_000   cl=3 dept=sales        [OK    ] PII=-
    #4 custom_002   cl=0 dept=all          [OK    ] PII=-
    #5 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=['client_id', 'email', 'person_name'] | MRR=1.00

  Δ Latency: +5.2ms (+62%) | Δ Breaches: 2 → 0 | Δ MRR: 1.00 → 1.00

  X1: Engineer cl=3 → IBAN (legal-only)
  Query: "IBAN bank account number for payments"
  User: cl=3, dept=engineering
  Intent: exact_critical (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 10.4ms median) ──
    #1 custom_004   cl=3 dept=legal        [BREACH] PII=iban, swift_code
    #2 custom_003   cl=2 dept=engineering  [OK    ] PII=-
    #3 custom_002   cl=0 dept=all          [OK    ] PII=-
    #4 custom_002   cl=3 dept=finance     


  ── SECURE ASYMMETRIC (5 chunks, 11.5ms median) ──
    #1 custom_003   cl=2 dept=engineering  [OK    ] PII=-
    #2 custom_002   cl=0 dept=all          [OK    ] PII=-
    #3 custom_001   cl=0 dept=all          [OK    ] PII=-
    #4 custom_002   cl=2 dept=engineering  [OK    ] PII=ip_address
    #5 custom_000   cl=2 dept=engineering  [OK    ] PII=ip_address
  Audit: PASS | breaches=0 | PII in results=['ip_address'] | MRR=0.00

  Δ Latency: +1.2ms (+11%) | Δ Breaches: 3 → 0 | Δ MRR: 1.00 → 0.00

  X2: Finance cl=3 → client emails (sales-only)
  Query: "email addresses of clients Laura Gomez and Marcos Ruiz"
  User: cl=3, dept=finance
  Intent: exact (α=0.2)

  ── UNSECURED BASELINE (5 chunks, 8.9ms median) ──
    #1 custom_003   cl=3 dept=sales        [BREACH] PII=email, client_id, person_name
    #2 custom_001   cl=3 dept=sales        [BREACH] PII=email, client_id, person_name
    #3 custom_000   cl=3 dept=sales        [BREACH] PII=-
    #4 custom_002   cl=3 dept=finance      [OK    ]


  ── SECURE ASYMMETRIC (5 chunks, 10.8ms median) ──
    #1 custom_002   cl=3 dept=finance      [OK    ] PII=monetary
    #2 custom_000   cl=3 dept=finance      [OK    ] PII=-
    #3 custom_003   cl=3 dept=finance      [OK    ] PII=-
    #4 custom_002   cl=0 dept=all          [OK    ] PII=-
    #5 custom_004   cl=3 dept=finance      [OK    ] PII=monetary
  Audit: PASS | breaches=0 | PII in results=['monetary'] | MRR=0.00

  Δ Latency: +1.9ms (+22%) | Δ Breaches: 3 → 0 | Δ MRR: 0.33 → 0.00

  P1: Public user → product info (cl=0)
  Query: "What are the contents of the Witty Kit?"
  User: cl=0, dept=all
  Intent: conceptual (α=0.8)

  ── UNSECURED BASELINE (5 chunks, 9.2ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_002   cl=3 dept=finance      [BREACH] PII=monetary
    #3 custom_000   cl=2 dept=engineering  [BREACH] PII=ip_address
    #4 custom_001   cl=0 dept=all          [OK    ] PII=-
    #5 custom_003   cl=2 dept=legal        [BREACH] PII=-
  A


  ── SECURE ASYMMETRIC (2 chunks, 9.3ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=[] | MRR=1.00

  Δ Latency: +0.1ms (+1%) | Δ Breaches: 3 → 0 | Δ MRR: 1.00 → 1.00

  P2: Public user → usage instructions (cl=0)
  Query: "How do I switch on the Witty timer?"
  User: cl=0, dept=all
  Intent: conceptual (α=0.8)

  ── UNSECURED BASELINE (5 chunks, 8.7ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_002   cl=2 dept=legal        [BREACH] PII=-
    #3 custom_003   cl=2 dept=legal        [BREACH] PII=-
    #4 custom_001   cl=0 dept=all          [OK    ] PII=-
    #5 custom_001   cl=3 dept=finance      [BREACH] PII=-
  Audit: FAIL | breaches=3 | leaked=[] | MRR=1.00



  ── SECURE ASYMMETRIC (2 chunks, 9.0ms median) ──
    #1 custom_002   cl=0 dept=all          [OK    ] PII=-
    #2 custom_001   cl=0 dept=all          [OK    ] PII=-
  Audit: PASS | breaches=0 | PII in results=[] | MRR=1.00

  Δ Latency: +0.3ms (+4%) | Δ Breaches: 3 → 0 | Δ MRR: 1.00 → 1.00



In [7]:
# Cell 7 — Summary tables + export

df = pd.DataFrame(all_results)

print("=" * 100)
print("  STEP 4: ASYMMETRIC HYBRID FUSION — RESULTS SUMMARY")
print("=" * 100)

# Security comparison
print("\n── Security Comparison ──")
sec_cols = ["tid", "description", "unsec_breaches", "sec_breaches", "unsec_security", "sec_security",
            "unsec_leaked_pii"]
print(df[sec_cols].to_string(index=False))

total_unsec_breaches = df["unsec_breaches"].sum()
total_sec_breaches = df["sec_breaches"].sum()
print(f"\n  Total breaches: UNSECURED={total_unsec_breaches} → SECURE={total_sec_breaches}")
print(f"  Breach elimination: {total_unsec_breaches - total_sec_breaches}/{total_unsec_breaches} "
      f"({(total_unsec_breaches - total_sec_breaches)/max(total_unsec_breaches,1)*100:.0f}%)")
sec_pass_unsec = (df["unsec_security"] == "PASS").sum()
sec_pass_sec = (df["sec_security"] == "PASS").sum()
print(f"  Security PASS rate: UNSECURED={sec_pass_unsec}/{len(df)} → SECURE={sec_pass_sec}/{len(df)}")

# Latency comparison
print("\n── Latency Comparison (median ms, {0} iterations) ──".format(N_BENCH))
lat_cols = ["tid", "description", "unsec_lat_ms", "sec_lat_ms", "lat_delta_ms", "lat_delta_pct"]
print(df[lat_cols].to_string(index=False))

avg_unsec = df["unsec_lat_ms"].mean()
avg_sec = df["sec_lat_ms"].mean()
avg_delta = df["lat_delta_ms"].mean()
print(f"\n  Average latency: UNSECURED={avg_unsec:.1f}ms → SECURE={avg_sec:.1f}ms "
      f"(Δ={avg_delta:+.1f}ms, {avg_delta/avg_unsec*100:+.0f}%)")

# MRR comparison
print("\n── MRR Comparison ──")
mrr_cols = ["tid", "description", "unsec_mrr", "sec_mrr"]
print(df[mrr_cols].to_string(index=False))
print(f"\n  Average MRR: UNSECURED={df['unsec_mrr'].mean():.3f} → SECURE={df['sec_mrr'].mean():.3f}")

# By category
for prefix, label in [("S", "Security tests (unauthorized)"),
                       ("A", "Authorized access tests"),
                       ("X", "Cross-department tests"),
                       ("P", "Public queries")]:
    sub = df[df["tid"].str.startswith(prefix)]
    if len(sub) == 0:
        continue
    print(f"\n  [{label}]")
    print(f"    Avg breaches: {sub['unsec_breaches'].mean():.1f} → {sub['sec_breaches'].mean():.1f}")
    print(f"    Avg latency:  {sub['unsec_lat_ms'].mean():.1f}ms → {sub['sec_lat_ms'].mean():.1f}ms "
          f"(Δ={sub['lat_delta_ms'].mean():+.1f}ms)")
    print(f"    Avg MRR:      {sub['unsec_mrr'].mean():.3f} → {sub['sec_mrr'].mean():.3f}")

# ── Export ──
csv_path = os.path.join(RESULTS_DIR, "ph2_step4_asymmetric_results.csv")
df.to_csv(csv_path, index=False)
print(f"\nExported: {csv_path}")

json_export = {
    "step": "Phase 2 - Step 4: Asymmetric Hybrid Fusion",
    "config": {
        "k": K, "bm25_pool": BM25_POOL, "n_warmup": N_WARMUP, "n_bench": N_BENCH,
        "corpus_size": len(corpus),
    },
    "aggregate": {
        "total_unsec_breaches": int(total_unsec_breaches),
        "total_sec_breaches": int(total_sec_breaches),
        "avg_lat_unsec_ms": round(avg_unsec, 2),
        "avg_lat_sec_ms": round(avg_sec, 2),
        "avg_lat_delta_ms": round(avg_delta, 2),
        "avg_mrr_unsec": round(df["unsec_mrr"].mean(), 4),
        "avg_mrr_sec": round(df["sec_mrr"].mean(), 4),
    },
    "results": all_results,
}
json_path = os.path.join(RESULTS_DIR, "ph2_step4_asymmetric_detail.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_export, f, indent=2, ensure_ascii=False)
print(f"Exported: {json_path}")

print("\nStep 4 complete. Awaiting approval before Step 5.")

  STEP 4: ASYMMETRIC HYBRID FUSION — RESULTS SUMMARY

── Security Comparison ──
tid                                 description  unsec_breaches  sec_breaches unsec_security sec_security                        unsec_leaked_pii
 S1   Public user → password (cl=3 engineering)               3             0           FAIL         PASS                                password
 S2             Public user → IBAN (cl=3 legal)               4             0           FAIL         PASS              iban, monetary, swift_code
 S3    Public user → client emails (cl=3 sales)               5             0           FAIL         PASS client_id, email, monetary, person_name
 S4 Public user → IP address (cl=2 engineering)               5             0           FAIL         PASS                              ip_address
 S5       Sales intern cl=1 → IBAN (cl=3 legal)               4             0           FAIL         PASS              iban, monetary, swift_code
 A1     Senior engineer → password (authoriz